In [1]:
# !!네이버 쇼핑몰 크롤링
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import WebDriverException
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
import undetected_chromedriver as uc
import time , tempfile 
import sys   
import math   
import pandas as pd   
import os
import random
import urllib.request
import urllib
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage

print('='*80)
print('네이버 쇼핑몰 검색 목록 크롤링 프로그램입니다.')
print('''
< 이 프로그램은 브랜드와 가격대별, 성별 검색한 상품에 대한 
다양한 순으로 데이터를 수집 할 수 있습니다. >
''')
print('='*80)

# 변수들
q = input('검색할 키워드는 무엇 입니까?(기본값: 암벽화): ')
if q == '':
    q = '암벽화'
qu = 'https://www.naver.com/'
cnt = input('수집할 데이터 건수는 몇 건입니까?(기본값: 50): ')
if cnt == '':
    cnt = 50
pcnt = math.ceil(int(cnt) / 10)

view_rank = input(
'''
원하시는 추천순은 무엇입니까(기본값: 추천순): 
1. 낮은 가격순, 2. 높은 가격순, 3. 판매 많은순, 4. 리뷰 많은순, 5. 신상품순
''')

minpri = input('원하시는 최소 가격은 얼마입니까?(기본값: 없음): ')
if minpri == '':
    pass

maxpri = input('원하시는 최대 가격은 얼마입니까?(기본값: 없음): ')
if maxpri == '':
    pass
    
now = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' %(now.tm_year, now.tm_mon, now.tm_mday, \
                                      now.tm_hour, now.tm_min, now.tm_sec)

ff = 'c:\\py_temp\\'
if ff == '':
    ff = 'c:\\py_temp\\'

img_dir = ff + s + '-' + q + '\\img'

ft_name = ff+s+'-'+q+'\\'+s+'-'+q+'.txt'
fx_name = ff+s+'-'+q+'\\'+s+'-'+q+'.xlsx'
fc_name = ff+s+'-'+q+'\\'+s+'-'+q+'.csv'

# 사이트 접속
print()
print('-'*80)
print('사이트에 접속중입니다~~~~ 잠시만 기다려 주세용~~^_^')
print('-'*80)

CHROME_PATH = r"C:\Program Files\Google\Chrome\Application\chrome.exe"  # 크롬 경로 확인!
VERSION_MAIN = 140  # 현재 크롬 메이저 버전

options = Options()
# 필요하면 headless: 많은 사이트가 탐지하니 처음엔 꺼두고 테스트
# options.add_argument("--headless=new")

# 깨끗한 임시 프로필(기존 프로필 충돌 방지)
user_data = os.path.join(tempfile.gettempdir(), "uc_profile_clean")
os.makedirs(user_data, exist_ok=True)

driver = uc.Chrome(
    options=options,
    browser_executable_path=CHROME_PATH,
    version_main=VERSION_MAIN,      # ← 핵심: 140으로 맞추기
    user_data_dir=user_data,
    use_subprocess=False,           # 안 되면 True로 바꿔 재시도
    patcher_force_close=True,       # 패쳐가 점유 중일 때 강제 종료
    debug=True,
)

driver.get(qu)
time.sleep(random.randrange(2,5))
driver.maximize_window()
time.sleep(random.randrange(2,5))

driver.find_element(By.XPATH,'//*[@id="shortcutArea"]/ul/li[4]').click( ) # 네이버 쇼핑몰 버튼
time.sleep(random.randrange(5,7))

driver.switch_to.window(driver.window_handles[-1]) # 현재 페이지 선택

element = driver.find_element(By.XPATH,'//*[@id="gnb-gnb"]/div[2]/div/div[2]/div[1]/form/div/div/div/div/input') # 검색창 버튼 선택
driver.find_element(By.XPATH,'//*[@id="gnb-gnb"]/div[2]/div/div[2]/div[1]/form/div/div/div/div/input').click( ) # 검색창 버튼 선택
for a in q:
    element.send_keys(a)
    time.sleep(random.randrange(1,3))
element.send_keys("\n")
time.sleep(random.randrange(3,5))

# 브랜드 검색
try:
    driver.find_element(By.XPATH,'//*[@id="LAYER_PRODUCT_FILTER_SCROLL_TARGET_ID"]/div/div/div[3]/div[2]/div[3]/div/div').click( ) # 브랜드 모두보기 버튼
    time.sleep(1)
        
    brand3 = [] # 브랜드
    bno = 1
    
    html = driver.page_source
    s = BeautifulSoup(html, 'html.parser')
    c1 = s.find('div', '_brand_search_filter_aside_pc_1rubw_12').find_all('span')
    time.sleep(1)
    
    print()
    print('-'*80)
    print('선택 가능한 브랜드는 다음과 같습니다.')
    
    for i in c1:
        c2 = i.get_text().strip()
        brand3.append(c2)
        brand3 = [x for x in brand3 if x != '']

    for br1 in brand3:
        print(str(bno) + '. ' + br1)
        bno += 1
    
    print()
    time.sleep(1)

    brand = input('원하시는 브랜드는 무엇입니까?(기본값: 전체) 원하는 브랜드 번호 누르기~~: ')
    if brand == '':
        pass

    else:
        driver.find_element(By.XPATH,'//*[@id="LAYER_PRODUCT_FILTER_SCROLL_TARGET_ID"]/div/div/div[3]/div[2]/div[3]/div/ul/li[%s]/div/label' %int(brand)).click( ) # 원하는 브랜드 버튼
        time.sleep(1)

except:
    pass
    print('브랜드가 없습니다.')

# 성별 선택
try:
    sex = input('원하시는 성별을 고르세요(1. 남녀공용, 2. 남성용, 3. 여성용): ')
    if sex == '':
        pass
    elif sex == '1':
        driver.find_element(By.XPATH,'//*[@id="FILTER_CHECKBOX_PC_ATTRIBUTE:10028204|M10030583"]').click( )
    elif sex == '2':
        driver.find_element(By.XPATH,'//*[@id="FILTER_CHECKBOX_PC_ATTRIBUTE:10028204|M10029065"]').click( )
    elif sex == '3':
        driver.find_element(By.XPATH,'//*[@id="FILTER_CHECKBOX_PC_ATTRIBUTE:10028204|M10537277"]').click( )

except:
    pass
    print('고를수 있는 성별이 없습니다.')

# 원하는 가격대 검색
print()
print('원하시는 가격대를 작성중입니다~~' + '\n')

try:
    minprice = driver.find_element(By.XPATH,'//*[@id="LAYER_PRODUCT_FILTER_SCROLL_TARGET_ID"]/div/div/div[7]/div[2]/div[1]/div[1]/label') # 최소 가격 버튼
    driver.find_element(By.XPATH,'//*[@id="LAYER_PRODUCT_FILTER_SCROLL_TARGET_ID"]/div/div/div[7]/div[2]/div[1]/div[1]/label').click( ) # 최소 가격 버튼
    for o in minpri:
        minprice.send_keys(o)
        time.sleep(random.randrange(1,3))
    
    time.sleep(random.randrange(3,5))
    
    maxprice = driver.find_element(By.XPATH,'//*[@id="LAYER_PRODUCT_FILTER_SCROLL_TARGET_ID"]/div/div/div[7]/div[2]/div[1]/div[3]/label') # 최대 가격 버튼
    driver.find_element(By.XPATH,'//*[@id="LAYER_PRODUCT_FILTER_SCROLL_TARGET_ID"]/div/div/div[7]/div[2]/div[1]/div[3]/label').click( ) # 최대 가격 버튼
    
    for p in maxpri:
        maxprice.send_keys(p)
        time.sleep(random.randrange(1,3))
    
    time.sleep(random.randrange(1,3))
    
    driver.find_element(By.XPATH,'//*[@id="LAYER_PRODUCT_FILTER_SCROLL_TARGET_ID"]/div/div/div[7]/div[2]/div[1]/button').click( ) # 검색 버튼    
    time.sleep(random.randrange(1,3))

except:
    pass

    
if view_rank == '': # 추천순
    pass
elif view_rank == '1':
    driver.find_element(By.XPATH,'//*[@id="product-sort-address-container"]/ul/li[2]/button').click( ) # 낮은 가격순 버튼
    time.sleep(random.randrange(3,5))
elif view_rank == '2':
    driver.find_element(By.XPATH,'//*[@id="product-sort-address-container"]/ul/li[3]/button').click( ) # 높은 가격순 버튼
    time.sleep(random.randrange(3,5))
elif view_rank == '3':
    driver.find_element(By.XPATH,'//*[@id="product-sort-address-container"]/ul/li[4]/button').click( ) # 판매 많은순 버튼
    time.sleep(random.randrange(3,5))
elif view_rank == '4':
    driver.find_element(By.XPATH,'//*[@id="product-sort-address-container"]/ul/li[5]/button').click( ) # 리뷰 많은순 버튼
    time.sleep(random.randrange(3,5))
elif view_rank == '5':
    driver.find_element(By.XPATH,'//*[@id="product-sort-address-container"]/ul/li[6]/button').click( ) # 신상순 버튼
    time.sleep(random.randrange(3,5))

def scroll_down(driver):      
    driver.execute_script("window.scrollBy(0,1000);")
    time.sleep(1)    

os.makedirs(img_dir)
os.chdir(img_dir)

no2 = [] # 번호
pstore2 = [] # 판매 가게
pname2 = [] # 상품 이름
pprice2 = [] # 가격
pdiscount2 = [] # 할인율
pcost2 = [] # 원가
pdeli2 = [] # 배송비
pstar2 = [] # 별점
priview2 = [] # 리뷰

no = 1
imgno = 1
pcno = 1

print('페이지를 학습중입니다~~~' + '\n')

for b in range(1,pcnt+1):
    scroll_down(driver)
    
    html = driver.page_source
    s = BeautifulSoup(html, 'html.parser')
    c1 = s.find('ul', 'compositeCardList_product_list__Ih4JR').find_all('li', 'compositeCardContainer_composite_card_container__jr8cb composite_card_container')

    pcno += 1

    if pcno > pcnt:
        break

print('이미지 저장을 시작합니다^_^>>>>>' + '\n')

for c in c1:
    os.chdir(img_dir)

    try:
        ad = c.find('span', 'promotionCard_badge__ZPa_x').get_text()
        if ad == '기획전':
            continue
            imgno -= 1
            
    except:
        try :
            imgc1 = c.find('div', 'productCardThumbnail_thumbnail__KzO1N').find('img')['src']
        except :
            continue
        imgc2 = imgc1 
        urllib.request.urlretrieve(imgc2,str(imgno)+'.jpg')
        print('%s 번째 이미지를 저장하고 있습니다.' %imgno)
    
        time.sleep(1)
        
        imgno += 1
    
        if imgno > int(cnt):
            break

print()
print('데이터 저장을 시작합니다^_^>>>>>' + '\n')

for d in c1:
    try:
        ad = d.find('span', 'promotionCard_badge__ZPa_x').get_text()
        if ad == '기획전':
            continue
            no -= 1
    except:
        no2.append(str(no) + '\n')
        print('1. 번호: ' + str(no))

    try:
        pname = d.find('strong', 'productCardTitle_product_card_title__eQupA productCardTitle_view_type_grid2__4N618').get_text()
    except:
        print('상품명이 없음')
        
    pname2.append(pname)
    print('2. 상품명: ' + pname)
    
    try:
        pstore = d.find('div', 'productCardMallLink_product_card_mall_link__H7GC2 productCardMallLink_view_type_grid2__ou5fk').find('span').get_text()
    except:
        print('판매가게가 없음')
        
    pstore2.append(pstore)
    print('3. 판매가게: ' + pstore)

    try:
        pcost = d.find('span', 'priceTag_original_price__jyZRY').get_text()[8:]
    except:
        pcost = d.find('span', 'priceTag_price_area__iHlni').get_text()
   
    pcost2.append(pcost)
    print('4. 원가: ' + pcost)

    try:
        pdiscount = d.find('span', 'priceTag_discount_ratio__VE866').get_text()[:3]
    except:
        pdiscount = '0%'

    pdiscount2.append(str(pdiscount))
    print('5. 할인율: ' + str(pdiscount))

    try:
        pprice = d.find('span', 'priceTag_price__hGtfm').get_text()
    except:
        pprice = d.find('span', 'priceTag_original_price__jyZRY').get_text()

    pprice2.append(pprice + '원')
    print('6. 판매가격: ' + pprice + '원')

    try:
        pdeli = d.find('span', 'productCardDeliveryFeeInfo_delivery_text__54pei').get_text()
    except:
        pdeli = d.find('div', 'productCardDeliveryBadge_badge__nvxV0').find('span').get_text()

    pdeli2.append(pdeli)
    print('7. 배송비: ' + pdeli)

    try:
        pstar = d.find('div', 'productCardReview_product_card_review__Oiv_T productCardReview_shown_review_text__XzE_x undefined').find('span', 'productCardReview_text__A9N9N').get_text()[2:]
    except:
        pstar = '0'
        pstar2.append(str(pstar) + '\n')
        print('8. 별점: ' + str(pstar))        
    else:
        pstar2.append(pstar + '\n')
        print('8. 별점: ' + pstar)

    try:
        priview = d.find_all('span', 'productCardReview_text__A9N9N')[1].get_text()[3:]
    except:
        priview = '0'
        priview2.append(str(priview) + '\n')
        print('9. 리뷰: ' + str(priview) + '\n')
    else:
        priview2.append(priview + '\n')
        print('9. 리뷰: ' + priview + '\n')

    no += 1

    if no > int(cnt):
        break

naver_shopping = pd.DataFrame()

naver_shopping['번호'] = no2 
naver_shopping['상품명'] = pd.Series(pname2)
naver_shopping['판매가게'] = pd.Series(pstore2)
naver_shopping['원가'] = pd.Series(pcost2)
naver_shopping['할인율'] = pd.Series(pdiscount2)
naver_shopping['판매 가격'] = pd.Series(pprice2)
naver_shopping['배송비'] = pd.Series(pdeli2)
naver_shopping['별점'] = pd.Series(pstar2)
naver_shopping['리뷰'] = pd.Series(priview2)

naver_shopping.to_csv(fc_name,encoding="utf-8-sig",index=False)
naver_shopping.to_excel(fx_name ,index=False)

print('='*80)
print('저장을 완료하였습니다.')
print('='*80)

wd = load_workbook(fx_name)
ws = wd["Sheet1"]

ws.column_dimensions['B'].width = 25
row_cnt = int(cnt) + 1

for r in range(2, row_cnt + 1):
    ws.row_dimensions[r].height = 100

for i in range(1, row_cnt):    
    cell_addr = f"B{i+1}"
    img_path = os.path.join(img_dir, f"{i}.jpg")
    if not os.path.exists(img_path):
        continue

    img = XLImage(img_path)
    img.width = 130
    img.height = 100

    ws.add_image(img, cell_addr)

wd.save(fx_name)
# driver.close()

print('='*80)
print('작업을 완료하였습니다. ^_^')

네이버 쇼핑몰 검색 목록 크롤링 프로그램입니다.

< 이 프로그램은 브랜드와 가격대별, 성별 검색한 상품에 대한 
다양한 순으로 데이터를 수집 할 수 있습니다. >



검색할 키워드는 무엇 입니까?(기본값: 암벽화):  
수집할 데이터 건수는 몇 건입니까?(기본값: 50):  50

원하시는 추천순은 무엇입니까(기본값: 추천순): 
1. 낮은 가격순, 2. 높은 가격순, 3. 판매 많은순, 4. 리뷰 많은순, 5. 신상품순
 3
원하시는 최소 가격은 얼마입니까?(기본값: 없음):  100000
원하시는 최대 가격은 얼마입니까?(기본값: 없음):  300000



--------------------------------------------------------------------------------
사이트에 접속중입니다~~~~ 잠시만 기다려 주세용~~^_^
--------------------------------------------------------------------------------

--------------------------------------------------------------------------------
선택 가능한 브랜드는 다음과 같습니다.
1. 매드락
2. 부토라
3. 스카르파
4. 라스포티바
5. 테나야
6. 블랙다이아몬드
7. 파이브텐
8. 이볼브
9. 보레알
10. 트랑고
11. 아디다스
12. 오순



원하시는 브랜드는 무엇입니까?(기본값: 전체) 원하는 브랜드 번호 누르기~~:  3
원하시는 성별을 고르세요(1. 남녀공용, 2. 남성용, 3. 여성용):  



원하시는 가격대를 작성중입니다~~

페이지를 학습중입니다~~~

이미지 저장을 시작합니다^_^>>>>>

1 번째 이미지를 저장하고 있습니다.
2 번째 이미지를 저장하고 있습니다.
3 번째 이미지를 저장하고 있습니다.
4 번째 이미지를 저장하고 있습니다.
5 번째 이미지를 저장하고 있습니다.
6 번째 이미지를 저장하고 있습니다.
7 번째 이미지를 저장하고 있습니다.
8 번째 이미지를 저장하고 있습니다.
9 번째 이미지를 저장하고 있습니다.
10 번째 이미지를 저장하고 있습니다.
11 번째 이미지를 저장하고 있습니다.
12 번째 이미지를 저장하고 있습니다.
13 번째 이미지를 저장하고 있습니다.
14 번째 이미지를 저장하고 있습니다.
15 번째 이미지를 저장하고 있습니다.
16 번째 이미지를 저장하고 있습니다.
17 번째 이미지를 저장하고 있습니다.
18 번째 이미지를 저장하고 있습니다.
19 번째 이미지를 저장하고 있습니다.
20 번째 이미지를 저장하고 있습니다.
21 번째 이미지를 저장하고 있습니다.
22 번째 이미지를 저장하고 있습니다.
23 번째 이미지를 저장하고 있습니다.
24 번째 이미지를 저장하고 있습니다.
25 번째 이미지를 저장하고 있습니다.
26 번째 이미지를 저장하고 있습니다.
27 번째 이미지를 저장하고 있습니다.
28 번째 이미지를 저장하고 있습니다.
29 번째 이미지를 저장하고 있습니다.
30 번째 이미지를 저장하고 있습니다.
31 번째 이미지를 저장하고 있습니다.
32 번째 이미지를 저장하고 있습니다.
33 번째 이미지를 저장하고 있습니다.
34 번째 이미지를 저장하고 있습니다.
35 번째 이미지를 저장하고 있습니다.
36 번째 이미지를 저장하고 있습니다.
37 번째 이미지를 저장하고 있습니다.
38 번째 이미지를 저장하고 있습니다.
39 번째 이미지를 저장하고 있습니다.
40 번째 이미지를 저장하고 있습니다.
41 번째 이미지를 저장하고 있습니다.
42 번째 이미지를 저장하고 있습니다.
43 번째 이미지를 저장하고 있습니다.
44 

['237,500원',
 '247,000원',
 '251,750원',
 '184,000원',
 '251,750원',
 '247,000원',
 '249,900원',
 '182,000원',
 '251,750원',
 '242,250원',
 '237,500원',
 '251,750원',
 '242,250원',
 '237,500원',
 '182,000원',
 '189,900원',
 '199,500원',
 '199,500원',
 '161,000원',
 '219,000원',
 '182,000원',
 '209,900원',
 '229,000원',
 '229,500원',
 '161,000원',
 '251,750원',
 '179,900원',
 '179,900원',
 '204,900원',
 '247,000원',
 '237,500원',
 '247,000원',
 '250,000원',
 '242,250원',
 '237,500원',
 '237,500원',
 '251,750원',
 '239,900원',
 '242,250원',
 '161,000원',
 '204,000원',
 '218,500원',
 '204,250원',
 '247,000원',
 '199,900원',
 '209,900원',
 '247,000원',
 '242,250원',
 '232,750원',
 '247,000원']